In [ ]:
import sys
sys.path.append("/home/vsokolov/polaris-hpc")

In [ ]:
from SALib.sample import morris as morris_s
from SALib.analyze import morris as morris_a
import SALib
import numpy as np
import os
import sys
# # import PolarisOpt
# from PolarisOpt.utils import archiver
# from PolarisOpt.F import build_sampleset
from PolarisOpt.setup_manager import SetupManager
from PolarisOpt.utils import objective_funcs


In [ ]:
import json

dictionary = json.loads(open('timedep_training_data.json').read())
for item in dictionary:
    print(item)

In [ ]:
#######################################
#######################################
# Pull in all of the settings for use #
#######################################
#######################################

settings_filepath = 'settings_slurm.json'
config_filepath = 'config_morris_transit.json'
manager=SetupManager(settings_filepath, config_filepath)


problem = {
    'num_vars': manager.dim_in,
    'names': manager.var,
    'bounds': manager.orig_range[0]
}
manager.get_dim_out()

In [ ]:
X,Err = manager.load_training()
Obj, _ = objective_funcs.run_objective(Err, o_type=manager.objective_type)
Si = morris_a.analyze(problem, X, Obj, conf_level=0.95, num_levels=4)
Si['names']

In [ ]:
import matplotlib.pyplot as plt
axes = Si.plot()
fig = plt.gcf()  # get current figure
fig.set_size_inches(10, 10)
plt.tight_layout()

# plt.tight_layout()
# plt.show()


# fig, ax = plt.subplots(1, 1)
# SALib.plotting.morris.horizontal_bar_plot(ax, Si,{}, sortby='mu_star', unit=r"tCO$_2$/year")
# horizontal_bar_plot(ax, Si,{}, sortby='mu_star', unit=r"tCO$_2$/year")

In [ ]:
plt.plot(np.sort(Obj[:,0])[::-1]); plt.xlabel('Rank'); plt.ylabel('Objective');

In [ ]:
np.argmin(Obj)

In [ ]:
X[:,10]

In [ ]:
np.where(np.array(Si['names'])=='FULLTIME_WORKER_WORK_ACTIVITY_FREQ')

In [ ]:
Si['names'][10]

In [ ]:
plt.scatter(X[:,10],Obj[:,0])

In [ ]:
 from sklearn.cross_decomposition import PLSRegression

In [ ]:
Y=Err
m = min(Y.shape[1],X.shape[1])
s = np.zeros(m-1) # sensitivity index
for i in range(1,m):
    pls = PLSRegression(n_components=i)
    pls.fit(X, Y)
    s[i-1] = pls.score(X, Y)

In [ ]:
plt.plot(s) 

In [ ]:
pls = PLSRegression(n_components=4)
pls.fit(X, Y);
pls.coef_.shape

In [ ]:
plt.bar(x=range(18),height=np.mean(np.abs(pls.coef_),axis=1))

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=114)
pca.fit(Y)

In [ ]:
plt.plot(pca.explained_variance_ratio_[0:10])

In [ ]:
plt.plot(np.arange(0,1440,2), np.mean(np.abs(Err),axis=0)); plt.xlabel('Time (min)'); plt.ylabel('Mean Error')